#upload dos Datasets Abide e Toddler

In [ ]:
import pandas as pd

df_abide = pd.read_csv('/content/ABIDEII_Long_Composite_Phenotypic.csv')
df_toddler_autism = pd.read_csv('/content/Toddler Autism dataset July 2018.csv')

print("ABIDEII_Long_Composite_Phenotypic.csv loaded into df_abide.")
print("Toddler Autism dataset July 2018.csv loaded into df_toddler_autism.")

ABIDEII_Long_Composite_Phenotypic.csv loaded into df_abide.
Toddler Autism dataset July 2018.csv loaded into df_toddler_autism.


In [ ]:
df_abide.head()


,SITE_ID,SUB_ID,SESSION,NDAR_GUID,DX_GROUP,PDD_DSM_IV_TR,ASD_DSM_5,AGE_AT_SCAN,SEX,HANDEDNESS_CATEGORY,...,ADI_R_C3_TOTAL,ADI_R_C4_REPETITIVE_USE_OBJECTS,ADI_R_C4_HIGHER,ADI_R_C4_UNUSUAL_SENSORY_INTERESTS,ADI_R_C4_TOTAL,ADI_R_D_AGE_PARENT_NOTICED,ADI_R_D_AGE_FIRST_SINGLE_WORDS,ADI_R_D_AGE_FIRST_PHRASES,ADI_R_D_AGE_WHEN_ABNORMALITY,ADI_R_D_INTERVIEWER_JUDGMENT
0,ABIDEI-PITT,50002,Baseline,NaN,1,1,NaN,16.77,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABIDEII-UPSM_Long,50002,Followup_1,NaN,1,1,NaN,18.49,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABIDEI-PITT,50005,Baseline,NaN,1,1,NaN,13.73,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABIDEII-UPSM_Long,50005,Followup_1,NaN,1,1,NaN,15.55,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABIDEI-PITT,50006,Baseline,NaN,1,1,NaN,13.37,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_toddler_autism.head()

,Case_No,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age_Mons,Qchat-10-Score,Sex,Ethnicity,Jaundice,Family_mem_with_ASD,Who completed the test,Class/ASD Traits
0,1,0,0,0,0,0,0,1,1,0,1,28,3,f,middle eastern,yes,no,family member,No
1,2,1,1,0,0,0,1,1,0,0,0,36,4,m,White European,yes,no,family member,Yes
2,3,1,0,0,0,0,0,1,1,0,1,36,4,m,middle eastern,yes,no,family member,Yes
3,4,1,1,1,1,1,1,1,1,1,1,24,10,m,Hispanic,no,no,family member,Yes
4,5,1,1,0,1,1,1,1,1,1,1,20,9,f,White European,no,yes,family member,Yes


#Primeiros Testes

In [ ]:
# !pip install -q statsmodels scikit-learn scipy


In [ ]:

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, train_test_split, cross_val_score

In [ ]:
def permutation_test_correlation(x, y, n_perm=5000, seed=42):
    """Teste de permutação para correlação entre uma feature binária/contínua e um rótulo binário."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    observed = np.corrcoef(x, y)[0, 1]
    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        y_perm = rng.permutation(y)
        perm_stats[i] = np.corrcoef(x, y_perm)[0, 1]
    p_value = (np.sum(np.abs(perm_stats) >= np.abs(observed)) + 1) / (n_perm + 1)
    return observed, p_value

def permutation_test_two_sample(a, b, n_perm=5000, seed=42):
    """Teste de permutação para diferença de médias entre dois grupos independentes."""
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(pooled)
        perm_stats[i] = perm[:n_a].mean() - perm[n_a:].mean()
    p_value = (np.sum(np.abs(perm_stats) >= np.abs(observed)) + 1) / (n_perm + 1)
    return observed, p_value

def apply_fdr(pvalues, alpha=0.05):
    reject, p_adj, _, _ = multipletests(pvalues, alpha=alpha, method='fdr_bh')
    return reject, p_adj

## Estudo de caso Dataset: Toddlers

In [ ]:
df = df_toddler_autism
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())

item_cols = [f"A{i}" for i in range(1, 11)]
score_col = [c for c in df.columns if "qchat" in c.lower() and "score" in c.lower()][0]
label_col = [c for c in df.columns if "class" in c.lower()][0]

print("Coluna de score:", score_col)
print("Coluna de rótulo:", label_col)

['Case_No', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age_Mons', 'Qchat-10-Score', 'Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who completed the test', 'Class/ASD Traits']
Coluna de score: Qchat-10-Score
Coluna de rótulo: Class/ASD Traits


Passo 1 — demonstração do vazamento pelo escore composto:



In [ ]:
df["Score_recomputado"] = df[item_cols].sum(axis=1)
df["Label_pela_regra"] = np.where(df["Score_recomputado"] > 3, "Yes", "No")

label_norm = df[label_col].astype(str).str.strip().str.lower()
regra_norm = df["Label_pela_regra"].str.lower()
concordancia = (label_norm == regra_norm).mean()
print(f"Concordância rótulo original vs. regra determinística (score>3): {concordancia*100:.2f}%")

y = (label_norm == "yes").astype(int).values

X_leak = df[[score_col]].values
clf_leak = LogisticRegression().fit(X_leak, y)
print(f"Acurácia usando só o Score (vazamento determinístico): {clf_leak.score(X_leak, y)*100:.2f}%")

Concordância rótulo original vs. regra determinística (score>3): 100.00%
Acurácia usando só o Score (vazamento determinístico): 100.00%


Passo 2, 3 e 4 — permutação item a item, FDR e seleção de features sem vazamento, validadas por holdout e LOOCV:



In [ ]:
resultados = []
for item in item_cols:
    r, p = permutation_test_correlation(df[item].values, y)
    resultados.append({"item": item, "correlacao": r, "p_valor": p})

res_df = pd.DataFrame(resultados)
reject, p_adj = apply_fdr(res_df["p_valor"].values)
res_df["p_adj_fdr"] = p_adj
res_df["significativo_fdr"] = reject
res_df = res_df.sort_values("correlacao", ascending=False).reset_index(drop=True)
print(res_df)

top_itens = res_df["item"].head(3).tolist()
print("Itens selecionados (sem Score):", top_itens)

X = df[top_itens].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
clf = LogisticRegression().fit(X_train, y_train)
acc_holdout = clf.score(X_test, y_test)
print(f"Acurácia holdout (25%) usando {len(top_itens)} itens: {acc_holdout*100:.2f}%")

loo_scores = cross_val_score(LogisticRegression(), X, y, cv=LeaveOneOut())
print(f"Acurácia média LOOCV: {loo_scores.mean()*100:.2f}%  (n={len(loo_scores)})")

  item  correlacao  p_valor  p_adj_fdr  significativo_fdr
0   A9    0.577336   0.0002     0.0002               True
1   A6    0.569424   0.0002     0.0002               True
2   A5    0.563297   0.0002     0.0002               True
3   A7    0.563177   0.0002     0.0002               True
4   A4    0.505204   0.0002     0.0002               True
5   A1    0.503810   0.0002     0.0002               True
6   A2    0.463467   0.0002     0.0002               True
7   A8    0.427155   0.0002     0.0002               True
8   A3    0.409701   0.0002     0.0002               True
9  A10    0.179833   0.0002     0.0002               True
Itens selecionados (sem Score): ['A9', 'A6', 'A5']
Acurácia holdout (25%) usando 3 itens: 88.26%
Acurácia média LOOCV: 87.19%  (n=1054)


## Estudo de caso Dataset: ABIDE

Estudo de caso A — ABIDE II Longitudinal (n=38 sujeitos, 2 sessões)



In [ ]:
df_abide.columns = [c.strip() for c in df_abide.columns]

numeric_candidates = [
    "AGE_AT_SCAN", "FIQ", "VIQ", "PIQ", "HANDEDNESS_SCORES",
    "SRS_TOTAL_RAW", "SCQ_TOTAL", "AQ_TOTAL",
    "ADOS_2_TOTAL", "ADOS_2_SEVERITY_TOTAL", "VINELAND_ABC_Standard"
]
numeric_candidates = [c for c in numeric_candidates if c in df_abide.columns]

for c in numeric_candidates:
    df_abide[c] = pd.to_numeric(df_abide[c], errors="coerce")
    df_abide.loc[df_abide[c] <= -9000, c] = np.nan

print("Variáveis numéricas disponíveis:", numeric_candidates)

Variáveis numéricas disponíveis: ['AGE_AT_SCAN', 'FIQ', 'VIQ', 'PIQ', 'HANDEDNESS_SCORES', 'SRS_TOTAL_RAW', 'SCQ_TOTAL', 'AQ_TOTAL', 'ADOS_2_TOTAL', 'ADOS_2_SEVERITY_TOTAL', 'VINELAND_ABC_Standard']


Diagnóstico de disponibilidade de dados por sessão (etapa importante antes de qualquer teste, já que nem toda variável clínica é reavaliada no follow-up):



In [ ]:
disponibilidade = (
    df_abide.groupby("SESSION")[numeric_candidates]
    .apply(lambda g: g.notna().sum())
)
print(disponibilidade)

            AGE_AT_SCAN  FIQ  VIQ  PIQ  HANDEDNESS_SCORES  SRS_TOTAL_RAW  \
SESSION                                                                    
Baseline             38   37   38   38                  0              0   
Followup_1           38    0    0    0                  0              0   

            SCQ_TOTAL  AQ_TOTAL  ADOS_2_TOTAL  ADOS_2_SEVERITY_TOTAL  \
SESSION                                                                
Baseline            0         0            14                     14   
Followup_1          0         0             0                      0   

            VINELAND_ABC_Standard  
SESSION                            
Baseline                        0  
Followup_1                      0  


Passo 2, 3 — permutação e FDR aplicados a múltiplas variáveis candidatas na linha de base (comparando grupo Autismo vs. Controle):



In [ ]:
baseline = df_abide[df_abide["SESSION"].str.contains("Baseline", case=False, na=False)]

resultados_baseline = []
for var in numeric_candidates:
    sub = baseline.dropna(subset=[var, "DX_GROUP"])
    g1 = sub.loc[sub["DX_GROUP"] == 1, var].values
    g2 = sub.loc[sub["DX_GROUP"] == 2, var].values
    if len(g1) < 3 or len(g2) < 3:
        continue
    diff, p_perm = permutation_test_two_sample(g1, g2)
    t_stat, p_classico = stats.ttest_ind(g1, g2, equal_var=False, nan_policy="omit")
    resultados_baseline.append({
        "variavel": var, "n_autismo": len(g1), "n_controle": len(g2),
        "diferenca_media": diff, "p_permutacao": p_perm, "p_ttest_classico": p_classico
    })

res_baseline_df = pd.DataFrame(resultados_baseline)
reject, p_adj = apply_fdr(res_baseline_df["p_permutacao"].values)
res_baseline_df["p_adj_fdr"] = p_adj
res_baseline_df["significativo_fdr"] = reject
print(res_baseline_df.sort_values("p_permutacao"))

      variavel  n_autismo  n_controle  diferenca_media  p_permutacao  \
3          PIQ         23          15       -10.344928      0.035193   
1          FIQ         22          15        -9.172727      0.067187   
2          VIQ         23          15        -7.281159      0.098780   
0  AGE_AT_SCAN         23          15        -0.818174      0.231954   

   p_ttest_classico  p_adj_fdr  significativo_fdr  
3          0.018641   0.131707              False  
1          0.048391   0.131707              False  
2          0.091752   0.131707              False  
0          0.214991   0.231954              False  


In [ ]:
def testar_estabilidade(var):
    pivot = df_abide.pivot_table(index="SUB_ID", columns="SESSION", values=var)
    pivot = pivot.dropna()

    # Ensure both 'Baseline' and 'Followup_1' columns exist after dropping NaNs
    if "Baseline" not in pivot.columns or "Followup_1" not in pivot.columns:
        return None

    # Check if there are enough pairs (subjects with data in both sessions)
    if pivot.shape[0] < 5:
        return None

    baseline_vals = pivot["Baseline"].values
    followup_vals = pivot["Followup_1"].values
    stat, p = stats.wilcoxon(baseline_vals, followup_vals)
    return {"variavel": var, "n_pares": pivot.shape[0], "estatistica_wilcoxon": stat, "p_valor": p}

estabilidade = [testar_estabilidade(v) for v in numeric_candidates]
estabilidade = [r for r in estabilidade if r is not None]
print(pd.DataFrame(estabilidade))

      variavel  n_pares  estatistica_wilcoxon       p_valor
0  AGE_AT_SCAN       38                   0.0  7.727609e-08


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. Isolar a linha de base (única sessão com FIQ/PIQ preenchidos)
baseline = df_abide[df_abide["SESSION"] == "Baseline"].copy()

# 2. Selecionar variáveis de interesse e remover ausentes
cols = ["DX_GROUP", "PIQ", "FIQ"]
baseline = baseline[cols].dropna()

# Tratar códigos de missing conhecidos do ABIDE (-9999)
baseline = baseline[(baseline["PIQ"] > 0) & (baseline["FIQ"] > 0)]

y = baseline["DX_GROUP"].values  # 1 = ASD, 2 = TD (conforme legenda do ABIDE)
X_piq = baseline[["PIQ"]].values
X_fiq = baseline[["FIQ"]].values

print(f"N final após limpeza: {len(y)}  (grupo 1: {(y==1).sum()}, grupo 2: {(y==2).sum()})")

def testar_esquemas_cv(X, y, nome_variavel, n_perm=500, random_state=42):
    modelo = make_pipeline(StandardScaler(), LogisticRegression())
    esquemas = {
        "k=3": KFold(n_splits=3, shuffle=True, random_state=random_state),
        "k=5": KFold(n_splits=5, shuffle=True, random_state=random_state),
        "LOOCV": LeaveOneOut(),
    }
    resultados = []
    for nome_esquema, cv in esquemas.items():
        acc_obs = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy").mean()

        # Distribuição nula: embaralhar rótulos e repetir o mesmo esquema de CV
        rng = np.random.RandomState(random_state)
        acc_null = []
        for _ in range(n_perm):
            y_perm = rng.permutation(y)
            acc_null.append(cross_val_score(modelo, X, y_perm, cv=cv, scoring="accuracy").mean())
        acc_null = np.array(acc_null)
        p_valor = (np.sum(acc_null >= acc_obs) + 1) / (n_perm + 1)

        resultados.append({
            "variavel": nome_variavel,
            "esquema_cv": nome_esquema,
            "acuracia_observada": round(acc_obs, 4),
            "acuracia_nula_media": round(acc_null.mean(), 4),
            "p_valor_permutacao": round(p_valor, 4),
            "significativo_5pct": p_valor < 0.05,
        })
    return resultados

resultados_piq = testar_esquemas_cv(X_piq, y, "PIQ")
resultados_fiq = testar_esquemas_cv(X_fiq, y, "FIQ")

tabela_final = pd.DataFrame(resultados_piq + resultados_fiq)
print(tabela_final)

N final após limpeza: 37  (grupo 1: 22, grupo 2: 15)
  variavel esquema_cv  acuracia_observada  acuracia_nula_media  \
0      PIQ        k=3              0.5962               0.5340   
1      PIQ        k=5              0.5679               0.5464   
2      PIQ      LOOCV              0.5405               0.5584   
3      FIQ        k=3              0.6261               0.5341   
4      FIQ        k=5              0.5750               0.5474   
5      FIQ      LOOCV              0.5405               0.5582   

   p_valor_permutacao  significativo_5pct  
0              0.1617               False  
1              0.4032               False  
2              0.7206               False  
3              0.0579               False  
4              0.2994               False  
5              0.6926               False  


# Seguindo

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut, permutation_test_score
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ---------- 1. Carregamento ----------
df = df_abide.copy()

# ---------- 2. Auditoria de valores-sentinela ----------
# Bases ABIDE costumam usar -9999 (e variações) como código de dado ausente
sentinelas = [-9999, -999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

registro_sentinelas = []
for col in numeric_cols:
    for s in sentinelas:
        n_ocorrencias = (df[col] == s).sum()
        if n_ocorrencias > 0:
            registro_sentinelas.append({
                'coluna': col,
                'valor_sentinela': s,
                'n_linhas_afetadas': int(n_ocorrencias)
            })

tabela_sentinelas = pd.DataFrame(registro_sentinelas)
print("=== Colunas numéricas com valores-sentinela ===")
if tabela_sentinelas.empty:
    print("Nenhuma sentinela encontrada nas colunas numéricas.")
else:
    print(tabela_sentinelas.sort_values('n_linhas_afetadas', ascending=False).to_string(index=False))

# Substitui sentinelas por NaN antes de qualquer análise subsequente
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinelas), col] = np.nan

# ---------- 3. Distribuição por site (Baseline) ----------
base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})  # 1 = ASD, 0 = TD

print("\n=== Sujeitos e balanço de classes por site (Baseline) ===")
dist_site = base.groupby('SITE_ID')['target'].agg(['count', 'sum'])
dist_site.columns = ['n_total', 'n_ASD']
dist_site['n_TD'] = dist_site['n_total'] - dist_site['n_ASD']
print(dist_site)

# ---------- 4. Teste leave-site-out para as variáveis clínicas ----------
candidatos = [c for c in ['FIQ', 'VIQ', 'PIQ'] if c in base.columns]
resultados_site = []

for var in candidatos:
    sub = base.dropna(subset=[var, 'target', 'SITE_ID'])
    grupos = sub['SITE_ID']
    n_grupos = grupos.nunique()

    if n_grupos < 2 or sub['target'].nunique() < 2:
        resultados_site.append({
            'variavel': var, 'n': len(sub), 'n_sites': n_grupos,
            'obs': 'sites ou classes insuficientes para leave-site-out'
        })
        continue

    X = sub[[var]].values
    y = sub['target'].values

    logo = LeaveOneGroupOut()
    modelo = make_pipeline(StandardScaler(), GaussianNB())

    score, perm_scores, p_valor = permutation_test_score(
        modelo, X, y, groups=grupos, cv=logo,
        n_permutations=1000, random_state=42, n_jobs=-1
    )

    resultados_site.append({
        'variavel': var, 'n': len(sub), 'n_sites': n_grupos,
        'acuracia_leave_site_out': round(score, 4),
        'p_perm': round(p_valor, 4)
    })

tabela_leave_site_out = pd.DataFrame(resultados_site)
print("\n=== Classificação com validação leave-site-out ===")
print(tabela_leave_site_out.to_string(index=False))

=== Colunas numéricas com valores-sentinela ===
Nenhuma sentinela encontrada nas colunas numéricas.

=== Sujeitos e balanço de classes por site (Baseline) ===
               n_total  n_ASD  n_TD
SITE_ID                            
ABIDEI-PITT         17      9     8
ABIDEI-UCLA_1       13     10     3
ABIDEI-UCLA_2        8      4     4

=== Classificação com validação leave-site-out ===
variavel  n  n_sites  acuracia_leave_site_out  p_perm
     FIQ 37        3                   0.5180  0.4615
     VIQ 38        3                   0.5543  0.2428
     PIQ 38        3                   0.6989  0.0060


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# 1) CARREGAMENTO
df = df_abide.copy()

# 2) AUDITORIA DE SENTINELAS (robusta)
# Sentinelas comuns em bases ABIDE/NDAR: -9999, -9998, 999, 9999
sentinel_values = [-9999, -9998, 999, 9999]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("=== Auditoria de sentinelas (valores conhecidos: -9999, -9998, 999, 9999) ===")
found_any = False
for col in numeric_cols:
    mask = df[col].isin(sentinel_values)
    if mask.any():
        found_any = True
        print(f"\nColuna: {col}  |  N ocorrências: {mask.sum()}")
        print(df.loc[mask, ["SITE_ID", "SUB_ID", "SESSION", col]])
if not found_any:
    print("Nenhum valor sentinela encontrado nas colunas numéricas (com base na lista checada).")

# 3) CONSTRUÇÃO DA AMOSTRA ANALÍTICA (Caso A: baseline, PIQ, DX_GROUP, SITE_ID)
baseline = df[df["SESSION"] == "Baseline"].copy()

# Limpeza de sentinelas na coluna PIQ (se existir) antes de seguir
if "PIQ" in baseline.columns:
    baseline["PIQ"] = baseline["PIQ"].replace(sentinel_values, np.nan)

analytic = baseline.dropna(subset=["PIQ", "DX_GROUP", "SITE_ID"]).copy()
analytic["y"] = (analytic["DX_GROUP"] == 1).astype(int)  # 1 = ASD, 0 = TD

print("\n=== N final da amostra analítica (Caso A, PIQ baseline) ===")
print(f"N = {len(analytic)}  |  Sites únicos = {analytic['SITE_ID'].nunique()}")

print("\n=== Distribuição por site (ASD vs TD) ===")
dist = analytic.groupby("SITE_ID")["y"].value_counts().unstack(fill_value=0)
dist.columns = ["TD (0)", "ASD (1)"] if 0 in dist.columns and 1 in dist.columns else dist.columns
print(dist)

# 4) LEAVE-SITE-OUT CV (classificador: regressão logística univariada em PIQ)
def leave_site_out_eval(data, feature_col="PIQ", label_col="y", site_col="SITE_ID"):
    sites = data[site_col].unique()
    y_true_all, y_pred_all = [], []

    for site in sites:
        train = data[data[site_col] != site]
        test = data[data[site_col] == site]

        # Precisa de pelo menos 2 classes no treino para treinar o classificador
        if train[label_col].nunique() < 2 or len(test) == 0:
            continue

        X_train = train[[feature_col]].values
        y_train = train[label_col].values
        X_test = test[[feature_col]].values
        y_test = test[label_col].values

        clf = LogisticRegression()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

    return np.array(y_true_all), np.array(y_pred_all)

y_true, y_pred = leave_site_out_eval(analytic)

acc = (y_true == y_pred).mean()

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan  # recall da classe ASD
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan  # recall da classe TD
balanced_acc = np.nanmean([sensitivity, specificity])

print("\n=== Resultado observado: Leave-Site-Out (PIQ -> DX_GROUP) ===")
print(f"Acurácia bruta       : {acc:.4f}")
print(f"Sensibilidade (ASD)  : {sensitivity:.4f}")
print(f"Especificidade (TD)  : {specificity:.4f}")
print(f"Acurácia balanceada  : {balanced_acc:.4f}")
print(f"Matriz de confusão [TN, FP, FN, TP]: {tn}, {fp}, {fn}, {tp}")

# 5) TESTE DE PERMUTAÇÃO (embaralhamento DENTRO de cada site, preservando proporção ASD/TD por site)
np.random.seed(42)
n_perm = 2000

perm_acc = np.zeros(n_perm)
perm_bal_acc = np.zeros(n_perm)

data_perm = analytic.copy()

for i in range(n_perm):
    # Embaralha 'y' dentro de cada SITE_ID (mantém a contagem de ASD/TD por site)
    data_perm["y_perm"] = (
        data_perm.groupby("SITE_ID")["y"]
        .transform(lambda x: np.random.permutation(x.values))
    )

    y_true_p, y_pred_p = leave_site_out_eval(
        data_perm.rename(columns={"y": "y_original", "y_perm": "y"}),
        feature_col="PIQ", label_col="y", site_col="SITE_ID"
    )

    if len(y_true_p) == 0:
        perm_acc[i] = np.nan
        perm_bal_acc[i] = np.nan
        continue

    perm_acc[i] = (y_true_p == y_pred_p).mean()

    cm_p = confusion_matrix(y_true_p, y_pred_p, labels=[0, 1])
    tn_p, fp_p, fn_p, tp_p = cm_p.ravel()
    sens_p = tp_p / (tp_p + fn_p) if (tp_p + fn_p) > 0 else np.nan
    spec_p = tn_p / (tn_p + fp_p) if (tn_p + fp_p) > 0 else np.nan
    perm_bal_acc[i] = np.nanmean([sens_p, spec_p])

# p-valor: proporção de permutações com estatística >= observada
p_value_acc = (np.nansum(perm_acc >= acc) + 1) / (np.sum(~np.isnan(perm_acc)) + 1)
p_value_bal_acc = (np.nansum(perm_bal_acc >= balanced_acc) + 1) / (np.sum(~np.isnan(perm_bal_acc)) + 1)

print("\n=== Teste de permutação (embaralhamento estratificado por site, N=2000) ===")
print(f"Acurácia observada        : {acc:.4f}  | p-valor: {p_value_acc:.4f}")
print(f"Acurácia balanceada obs.  : {balanced_acc:.4f}  | p-valor: {p_value_bal_acc:.4f}")
print(f"Acurácia média sob H0     : {np.nanmean(perm_acc):.4f}")
print(f"Acurácia balanceada média sob H0 : {np.nanmean(perm_bal_acc):.4f}")

=== Auditoria de sentinelas (valores conhecidos: -9999, -9998, 999, 9999) ===
Nenhum valor sentinela encontrado nas colunas numéricas (com base na lista checada).

=== N final da amostra analítica (Caso A, PIQ baseline) ===
N = 38  |  Sites únicos = 3

=== Distribuição por site (ASD vs TD) ===
               TD (0)  ASD (1)
SITE_ID                       
ABIDEI-PITT         8        9
ABIDEI-UCLA_1       3       10
ABIDEI-UCLA_2       4        4

=== Resultado observado: Leave-Site-Out (PIQ -> DX_GROUP) ===
Acurácia bruta       : 0.6053
Sensibilidade (ASD)  : 0.7826
Especificidade (TD)  : 0.3333
Acurácia balanceada  : 0.5580
Matriz de confusão [TN, FP, FN, TP]: 5, 10, 5, 18

=== Teste de permutação (embaralhamento estratificado por site, N=2000) ===
Acurácia observada        : 0.6053  | p-valor: 0.1769
Acurácia balanceada obs.  : 0.5580  | p-valor: 0.0710
Acurácia média sob H0     : 0.5323
Acurácia balanceada média sob H0 : 0.4618


In [ ]:
import pandas as pd
import numpy as np

# Carregar o arquivo CSV
# Certifique-se de que o arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' está no ambiente do Colab
try:
    df = df_abide.copy()
    print("Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.")
except FileNotFoundError:
    print("ERRO: O arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto do Colab.")
    exit()

print("\n=== Todas as colunas disponíveis no dataset ABIDE II Longitudinal ===")
for col in df.columns:
    print(col)

print("\n=== Tentativa de identificação de colunas relacionadas a SRS-2, ADOS e QI ===")

# Termos de busca comuns para SRS-2, ADOS e QI
search_terms = ['SRS', 'ADOS', 'IQ_TEST', 'IQ_TYPE', 'WASI', 'WISC', 'DAS', 'FIQ_TEST', 'VIQ_TEST', 'PIQ_TEST']
found_cols = {}

for term in search_terms:
    for col in df.columns:
        if term.lower() in col.lower():
            if col not in found_cols:
                found_cols[col] = term

if not found_cols:
    print("Nenhuma coluna identificada com os termos de busca comuns.")
else:
    for col, term in found_cols.items():
        print(f"\nColuna identificada: {col} (termo: {term})")
        print(f"Tipo de dado: {df[col].dtype}")
        # Mostra os 5 primeiros valores não nulos para ter uma ideia do conteúdo
        sample_values = df[col].dropna().head(5).tolist()
        if sample_values:
            print(f"Exemplo de valores: {sample_values}")
        else:
            print("Coluna vazia ou sem valores não nulos.")

print("\nPor favor, revise a lista acima e confirme os nomes exatos das colunas que correspondem a:")
print("1. Escalas SRS-2 (escore total e/ou subescalas)")
print("2. Escalas ADOS (escore total e/ou subescalas)")
print("3. Tipo de teste de QI (se houver uma coluna específica para isso)")
print("Com essas informações, podemos prosseguir com as análises.")

Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.

=== Todas as colunas disponíveis no dataset ABIDE II Longitudinal ===
SITE_ID
SUB_ID
SESSION
NDAR_GUID
DX_GROUP
PDD_DSM_IV_TR
ASD_DSM_5
AGE_AT_SCAN
SEX
HANDEDNESS_CATEGORY
HANDEDNESS_SCORES
FIQ
VIQ
PIQ
FIQ_TEST_TYPE
VIQ_TEST_TYPE
PIQ_TEST_TYPE
ADI_R_SOCIAL_TOTAL_A
ADI_R_VERBAL_TOTAL_BV
ADI_R_NONVERBAL_TOTAL_BV
ADI_R_RRB_TOTAL_C
ADI_R_ONSET_TOTAL_D
ADI_R_RSRCH_RELIABLE
ADOS_MODULE
ADOS_RSRCH_RELIABLE
ADOS_G_TOTAL
ADOS_G_COMM
ADOS_G_SOCIAL
ADOS_G_STEREO_BEHAV
ADOS_G_CREATIVITY
ADOS_2_SOCAFFECT
ADOS_2_RRB
ADOS_2_TOTAL
ADOS_2_SEVERITY_TOTAL
SRS_EDITION
SRS_VERSION
SRS_INFORMANT
SRS_TOTAL_RAW
SRS_AWARENESS_RAW
SRS_COGNITION_RAW
SRS_COMMUNICATION_RAW
SRS_MOTIVATION_RAW
SRS_MANNERISMS_RAW
SRS_TOTAL_T
SRS_AWARENESS_T
SRS_COGNITION_T
SRS_COMMUNICATION_T
SRS_MOTIVATION_T
SRS_MANNERISMS_T
SCQ_VERSION
SCQ_TOTAL
AQ_TOTAL
NONASD_PSYDX_ICD9CODE
NONASD_PSYDX_LABEL
CURRENT_MED_STATUS
CURRENT_MEDICATION_NAME
OFF_STIMULANTS_AT_SCAN
E

# ABIDE ADOS2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut

# --- Funções Auxiliares (reutilizadas do código anterior) ---

def permutation_test_group_difference(data, feature_col, group_col, n_permutations=1000):
    """
    Realiza um teste de permutação para a diferença de médias entre dois grupos.
    Assume que group_col tem exatamente dois valores únicos.
    """
    groups = data[group_col].unique()
    if len(groups) != 2:
        raise ValueError("A coluna de grupo deve ter exatamente dois valores únicos.")

    group1_data = data[data[group_col] == groups[0]][feature_col].dropna()
    group2_data = data[data[group_col] == groups[1]][feature_col].dropna()

    if group1_data.empty or group2_data.empty:
        return np.nan, np.nan, np.nan, np.nan # Retorna NaN se um dos grupos estiver vazio

    observed_diff = np.mean(group1_data) - np.mean(group2_data)

    pooled_data = data[[feature_col, group_col]].dropna()
    if pooled_data.empty:
        return np.nan, np.nan, np.nan, np.nan

    permutation_diffs = []
    for _ in range(n_permutations):
        perm_labels = np.random.permutation(pooled_data[group_col].values)
        perm_group1_data = pooled_data[feature_col][perm_labels == groups[0]]
        perm_group2_data = pooled_data[feature_col][perm_labels == groups[1]]

        if not perm_group1_data.empty and not perm_group2_data.empty:
            permutation_diffs.append(np.mean(perm_group1_data) - np.mean(perm_group2_data))
        else:
            # Se uma permutação resultar em grupo vazio, ignorar ou tratar como NaN
            permutation_diffs.append(np.nan)

    permutation_diffs = np.array(permutation_diffs)
    permutation_diffs = permutation_diffs[~np.isnan(permutation_diffs)] # Remover NaNs

    if not permutation_diffs.size: # Se todas as permutações resultaram em NaN
        return observed_diff, np.nan, np.nan, np.nan

    p_value = (np.sum(np.abs(permutation_diffs) >= np.abs(observed_diff)) + 1) / (len(permutation_diffs) + 1)

    return observed_diff, p_value, np.mean(group1_data), np.mean(group2_data)


def leave_site_out_eval(data, feature_col, label_col, site_col):
    """
    Realiza a avaliação Leave-Site-Out e retorna y_true e y_pred.
    """
    sites = data[site_col].unique()
    y_true_all, y_pred_all = [], []

    for site in sites:
        train = data[data[site_col] != site]
        test = data[data[site_col] == site]

        if train[label_col].nunique() < 2 or len(test) == 0:
            continue

        X_train = train[[feature_col]].values
        y_train = train[label_col].values
        X_test = test[[feature_col]].values
        y_test = test[label_col].values

        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

    return np.array(y_true_all), np.array(y_pred_all)

# --- 1. Carregamento e Limpeza de Sentinelas ---
try:
    df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
    print("Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.")
except FileNotFoundError:
    print("ERRO: O arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto do Colab.")
    exit()

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

# --- 2. Preparação da Amostra Analítica (Baseline) ---
base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})  # 1 = ASD, 0 = TD

# --- 3. Análise dos Escores ADOS-2 ---
ados_cols = ['ADOS_2_TOTAL', 'ADOS_2_SOCAFFECT', 'ADOS_2_RRB', 'ADOS_2_SEVERITY_TOTAL']
results_ados = []

print("\n=== Análise ADOS-2: Comparação de Grupo (ASD vs TD) e Classificação Leave-Site-Out ===")

for col in ados_cols:
    print(f"\n--- Variável: {col} ---")

    # Filtrar dados para a coluna ADOS atual
    current_data = base.dropna(subset=[col, 'target', 'SITE_ID'])

    if current_data.empty or current_data['target'].nunique() < 2:
        print(f"Dados insuficientes para {col} após remoção de NaNs ou apenas uma classe presente.")
        results_ados.append({
            'variavel': col, 'n_sujeitos': len(current_data),
            'obs': 'Dados insuficientes para análise.'
        })
        continue

    # --- Teste de Permutação para Diferença de Grupo ---
    obs_diff, p_val_diff, mean_g1, mean_g2 = permutation_test_group_difference(
        current_data, col, 'DX_GROUP', n_permutations=2000
    )

    if not np.isnan(p_val_diff):
        print(f"Diferença de Grupo (ASD vs TD):")
        print(f"  Média ASD: {mean_g1:.2f}, Média TD: {mean_g2:.2f}")
        print(f"  Diferença observada (ASD - TD): {obs_diff:.2f}")
        print(f"  P-valor (permutação): {p_val_diff:.4f}")
    else:
        print("  Não foi possível calcular a diferença de grupo (dados insuficientes).")

    # --- Classificação Leave-Site-Out ---
    y_true, y_pred = leave_site_out_eval(current_data, col, 'target', 'SITE_ID')

    if len(y_true) == 0:
        print("  Não foi possível realizar a classificação Leave-Site-Out (dados insuficientes).")
        results_ados.append({
            'variavel': col, 'n_sujeitos': len(current_data),
            'obs': 'Dados insuficientes para LSO-CV.'
        })
        continue

    acc = (y_true == y_pred).mean()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    balanced_acc = np.nanmean([sensitivity, specificity])

    print(f"Classificação Leave-Site-Out (N={len(y_true)}):")
    print(f"  Acurácia Bruta: {acc:.4f}")
    print(f"  Acurácia Balanceada: {balanced_acc:.4f}")
    print(f"  Sensibilidade (ASD): {sensitivity:.4f}")
    print(f"  Especificidade (TD): {specificity:.4f}")

    # --- Teste de Permutação para LSO-CV (estratificado por site) ---
    n_perm = 2000
    perm_acc = np.zeros(n_perm)
    perm_bal_acc = np.zeros(n_perm)

    data_perm = current_data.copy()

    for i in range(n_perm):
        data_perm["y_perm"] = (
            data_perm.groupby("SITE_ID")["target"]
            .transform(lambda x: np.random.permutation(x.values))
        )

        y_true_p, y_pred_p = leave_site_out_eval(
            data_perm.rename(columns={"target": "y_original", "y_perm": "target"}),
            feature_col=col, label_col="target", site_col="SITE_ID"
        )

        if len(y_true_p) == 0:
            perm_acc[i] = np.nan
            perm_bal_acc[i] = np.nan
            continue

        perm_acc[i] = (y_true_p == y_pred_p).mean()
        cm_p = confusion_matrix(y_true_p, y_pred_p, labels=[0, 1])
        tn_p, fp_p, fn_p, tp_p = cm_p.ravel()
        sens_p = tp_p / (tp_p + fn_p) if (tp_p + fn_p) > 0 else np.nan
        spec_p = tn_p / (tn_p + fp_p) if (tn_p + fp_p) > 0 else np.nan
        perm_bal_acc[i] = np.nanmean([sens_p, spec_p])

    p_value_acc = (np.nansum(perm_acc >= acc) + 1) / (np.sum(~np.isnan(perm_acc)) + 1)
    p_value_bal_acc = (np.nansum(perm_bal_acc >= balanced_acc) + 1) / (np.sum(~np.isnan(perm_bal_acc)) + 1)

    print(f"  P-valor (permutação, acurácia bruta): {p_value_acc:.4f}")
    print(f"  P-valor (permutação, acurácia balanceada): {p_value_bal_acc:.4f}")

    results_ados.append({
        'variavel': col, 'n_sujeitos': len(current_data),
        'n_sites': current_data['SITE_ID'].nunique(),
        'mean_ASD': mean_g1, 'mean_TD': mean_g2, 'diff_p_value': p_val_diff,
        'acc_bruta': acc, 'acc_balanceada': balanced_acc,
        'sensibilidade': sensitivity, 'especificidade': specificity,
        'lso_acc_p_value': p_value_acc, 'lso_bal_acc_p_value': p_value_bal_acc
    })

# --- 4. Análise do Tipo de Teste de QI ---
print("\n=== Análise do Tipo de Teste de QI ===")
iq_test_type_cols = ['FIQ_TEST_TYPE', 'VIQ_TEST_TYPE', 'PIQ_TEST_TYPE']

for col in iq_test_type_cols:
    print(f"\n--- Coluna: {col} ---")
    if col in base.columns:
        # Distribuição geral
        print("Distribuição geral:")
        print(base[col].value_counts(dropna=False))

        # Distribuição por site
        print("\nDistribuição por site:")
        print(base.groupby('SITE_ID')[col].value_counts(dropna=False).unstack(fill_value=0))

        # Distribuição por grupo diagnóstico
        print("\nDistribuição por grupo diagnóstico:")
        print(base.groupby('DX_GROUP')[col].value_counts(dropna=False).unstack(fill_value=0))
    else:
        print(f"Coluna '{col}' não encontrada no dataset base.")


Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.

=== Análise ADOS-2: Comparação de Grupo (ASD vs TD) e Classificação Leave-Site-Out ===

--- Variável: ADOS_2_TOTAL ---
Dados insuficientes para ADOS_2_TOTAL após remoção de NaNs ou apenas uma classe presente.

--- Variável: ADOS_2_SOCAFFECT ---
Dados insuficientes para ADOS_2_SOCAFFECT após remoção de NaNs ou apenas uma classe presente.

--- Variável: ADOS_2_RRB ---
Dados insuficientes para ADOS_2_RRB após remoção de NaNs ou apenas uma classe presente.

--- Variável: ADOS_2_SEVERITY_TOTAL ---
Dados insuficientes para ADOS_2_SEVERITY_TOTAL após remoção de NaNs ou apenas uma classe presente.

=== Análise do Tipo de Teste de QI ===

--- Coluna: FIQ_TEST_TYPE ---
Distribuição geral:
FIQ_TEST_TYPE
WASI       26
WISC-IV    12
Name: count, dtype: int64

Distribuição por site:
FIQ_TEST_TYPE  WASI  WISC-IV
SITE_ID                     
ABIDEI-PITT      17        0
ABIDEI-UCLA_1     4        9
ABIDEI-UCLA_2     5        3

Di

In [ ]:
import pandas as pd
import numpy as np

# --- Carregamento e Limpeza de Sentinelas (reutilizado) ---
try:
    df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
    print("Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.")
except FileNotFoundError:
    print("ERRO: O arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto do Colab.")
    exit()

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

# --- Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ---
print("\n=== Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ===")

# Excluir colunas já analisadas ou identificadas como problemáticas/confundidas
# e colunas de identificação/grupo
exclude_cols = [
    'SITE_ID', 'SUB_ID', 'SESSION', 'NDAR_GUID', 'DX_GROUP', 'PDD_DSM_IV_TR', 'ASD_DSM_5',
    'AGE_AT_SCAN', # Já analisada a mudança de idade
    'FIQ', 'VIQ', 'PIQ', # Confoundidas pelo tipo de teste
    'ADOS_2_TOTAL', 'ADOS_2_SOCAFFECT', 'ADOS_2_RRB', 'ADOS_2_SEVERITY_TOTAL', # Ausentes
    'SRS_TOTAL_RAW', 'SRS_AWARENESS_RAW', 'SRS_COGNITION_RAW', 'SRS_COMMUNICATION_RAW',
    'SRS_MOTIVATION_RAW', 'SRS_MANNERISMS_RAW', 'SRS_TOTAL_T', 'SRS_AWARENESS_T',
    'SRS_COGNITION_T', 'SRS_COMMUNICATION_T', 'SRS_MOTIVATION_T', 'SRS_MANNERISMS_T' # Ausentes
]

# Filtrar apenas colunas numéricas que não estão na lista de exclusão
potential_longitudinal_cols = [col for col in numeric_cols if col not in exclude_cols]

# Criar um DataFrame pivotado para facilitar a comparação entre sessões
# Usamos 'SUB_ID' para identificar os sujeitos únicos
df_pivot = df_limpo.pivot_table(index='SUB_ID', columns='SESSION', values=potential_longitudinal_cols)

print("Verificando variáveis numéricas com dados em ambas as sessões (Baseline e Follow-up):")
print("------------------------------------------------------------------------------------")

found_longitudinal_data = {}

for col in potential_longitudinal_cols:
    # Selecionar as colunas de baseline e follow-up para a variável atual
    baseline_data = df_pivot[col, 'Baseline']
    followup_data = df_pivot[col, 'Followup_1']

    # Contar quantos sujeitos têm dados não nulos em AMBAS as sessões para esta variável
    has_data_in_both = (~baseline_data.isna()) & (~followup_data.isna())
    num_subjects_with_both = has_data_in_both.sum()

    if num_subjects_with_both > 0:
        found_longitudinal_data[col] = num_subjects_with_both
        print(f"- {col}: {num_subjects_with_both} sujeitos com dados em Baseline e Follow-up.")

if not found_longitudinal_data:
    print("Nenhuma outra variável numérica encontrada com dados em ambas as sessões para múltiplos sujeitos.")
else:
    print("\nVariáveis encontradas com dados longitudinais:")
    for col, count in found_longitudinal_data.items():
        print(f"- {col} ({count} sujeitos)")


Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.

=== Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ===
Verificando variáveis numéricas com dados em ambas as sessões (Baseline e Follow-up):
------------------------------------------------------------------------------------
- AGE_AT_SCAN : 38 sujeitos com dados em Baseline e Follow-up.
- SEX: 38 sujeitos com dados em Baseline e Follow-up.
- HANDEDNESS_CATEGORY: 38 sujeitos com dados em Baseline e Follow-up.


KeyError: ('HANDEDNESS_SCORES', 'Baseline')

In [ ]:


import pandas as pd
import numpy as np

# --- Carregamento e Limpeza de Sentinelas (reutilizado) ---
try:
    df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
    print("Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.")
except FileNotFoundError:
    print("ERRO: O arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto do Colab.")
    exit()

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

# --- DEBUG: Identificar nomes de sessão ---
print("\n=== Verificando nomes únicos na coluna 'SESSION' ===")
unique_sessions = df_limpo['SESSION'].unique()
print(unique_sessions)

print("\nO erro 'KeyError: 'Follow-up'' indica que o nome da sessão de acompanhamento não é 'Follow-up'.")
print("Pela saída acima, por favor, identifique qual é o nome correto da sessão de 'Follow-up'.")
print("Por exemplo, pode ser 'Follow-up 1', 'followup_1', 'session_2', etc.")
print("Com essa informação, podemos corrigir o código e prosseguir.")



Arquivo 'ABIDEII_Long_Composite_Phenotypic.csv' carregado com sucesso.

=== Verificando nomes únicos na coluna 'SESSION' ===
['Baseline' 'Followup_1']

O erro 'KeyError: 'Follow-up'' indica que o nome da sessão de acompanhamento não é 'Follow-up'.
Pela saída acima, por favor, identifique qual é o nome correto da sessão de 'Follow-up'.
Por exemplo, pode ser 'Follow-up 1', 'followup_1', 'session_2', etc.
Com essa informação, podemos corrigir o código e prosseguir.


In [ ]:

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

# Nota: a coluna AGE_AT_SCAN tem um espaço em branco no nome original ('AGE_AT_SCAN ')
# Vamos padronizar os nomes das colunas removendo espaços em branco extras
df_limpo.columns = [c.strip() for c in df_limpo.columns]
numeric_cols = [c.strip() for c in numeric_cols]

# --- Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ---
print("\n=== Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ===")

BASELINE_LABEL = 'Baseline'
FOLLOWUP_LABEL = 'Followup_1'

# Excluir colunas já analisadas ou identificadas como problemáticas/confundidas
# e colunas de identificação/grupo
exclude_cols = [
    'SITE_ID', 'SUB_ID', 'SESSION', 'NDAR_GUID', 'DX_GROUP', 'PDD_DSM_IV_TR', 'ASD_DSM_5',
    'AGE_AT_SCAN',  # Já analisada a mudança de idade
    'FIQ', 'VIQ', 'PIQ',  # Confundidas pelo tipo de teste
    'ADOS_2_TOTAL', 'ADOS_2_SOCAFFECT', 'ADOS_2_RRB', 'ADOS_2_SEVERITY_TOTAL',  # Ausentes
    'SRS_TOTAL_RAW', 'SRS_AWARENESS_RAW', 'SRS_COGNITION_RAW', 'SRS_COMMUNICATION_RAW',
    'SRS_MOTIVATION_RAW', 'SRS_MANNERISMS_RAW', 'SRS_TOTAL_T', 'SRS_AWARENESS_T',
    'SRS_COGNITION_T', 'SRS_COMMUNICATION_T', 'SRS_MOTIVATION_T', 'SRS_MANNERISMS_T'  # Ausentes
]

# Filtrar apenas colunas numéricas que não estão na lista de exclusão
potential_longitudinal_cols = [col for col in numeric_cols if col not in exclude_cols]

# Criar um DataFrame pivotado para facilitar a comparação entre sessões
df_pivot = df_limpo.pivot_table(index='SUB_ID', columns='SESSION', values=potential_longitudinal_cols)

print("Verificando variáveis numéricas com dados em ambas as sessões (Baseline e Followup_1):")
print("---------------------------------------------------------------------------------------")

found_longitudinal_data = {}

for col in potential_longitudinal_cols:
    try:
        baseline_data = df_pivot[col, BASELINE_LABEL]
        followup_data = df_pivot[col, FOLLOWUP_LABEL]
    except KeyError:
        # Coluna pode não existir para uma das sessões
        continue

    has_data_in_both = (~baseline_data.isna()) & (~followup_data.isna())
    num_subjects_with_both = has_data_in_both.sum()

    if num_subjects_with_both > 0:
        found_longitudinal_data[col] = num_subjects_with_both

# Ordenar por número de sujeitos disponíveis, decrescente
found_longitudinal_data = dict(
    sorted(found_longitudinal_data.items(), key=lambda x: x[1], reverse=True)
)

if not found_longitudinal_data:
    print("Nenhuma outra variável numérica encontrada com dados em ambas as sessões para múltiplos sujeitos.")
else:
    print(f"\n{len(found_longitudinal_data)} variáveis encontradas com dados em ambas as sessões:")
    for col, count in found_longitudinal_data.items():
        print(f"- {col}: {count} sujeitos")



=== Análise Longitudinal: Identificação de Variáveis com Dados em Ambas as Sessões ===
Verificando variáveis numéricas com dados em ambas as sessões (Baseline e Followup_1):
---------------------------------------------------------------------------------------

6 variáveis encontradas com dados em ambas as sessões:
- SEX: 38 sujeitos
- HANDEDNESS_CATEGORY: 38 sujeitos
- CURRENT_MED_STATUS: 38 sujeitos
- EYE_STATUS_AT_SCAN: 38 sujeitos
- AGE_AT_MPRAGE: 21 sujeitos
- OFF_STIMULANTS_AT_SCAN: 1 sujeitos


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

BASELINE_LABEL = 'Baseline'
FOLLOWUP_LABEL = 'Followup_1'

df_pivot = df_limpo.pivot_table(
    index='SUB_ID',
    columns='SESSION',
    values=['SEX', 'HANDEDNESS_CATEGORY', 'CURRENT_MED_STATUS', 'EYE_STATUS_AT_SCAN'],
    aggfunc='first'
)

# --- 1. Checagem de consistência (SEX e HANDEDNESS_CATEGORY) ---
print("=== Checagem de Consistência entre Sessões (variáveis que não deveriam mudar) ===")
for col in ['SEX', 'HANDEDNESS_CATEGORY']:
    sub = df_pivot[col].dropna()
    if sub.empty:
        print(f"{col}: sem dados suficientes.")
        continue
    inconsistent = (sub[BASELINE_LABEL] != sub[FOLLOWUP_LABEL]).sum()
    total = len(sub)
    print(f"{col}: {inconsistent}/{total} sujeitos com valores diferentes entre Baseline e Followup_1.")

# --- 2. Teste de McNemar (CURRENT_MED_STATUS e EYE_STATUS_AT_SCAN) ---
print("\n=== Teste de McNemar (mudança entre sessões, variáveis categóricas) ===")
for col in ['CURRENT_MED_STATUS', 'EYE_STATUS_AT_SCAN']:
    sub = df_pivot[col].dropna()
    if sub.empty or sub[BASELINE_LABEL].nunique() < 2:
        print(f"{col}: dados insuficientes ou variável constante, não é possível aplicar McNemar.")
        continue

    # Construir tabela de contingência 2x2 (assume variável binária; se não for, reportar categorias)
    categories = sorted(set(sub[BASELINE_LABEL].unique()) | set(sub[FOLLOWUP_LABEL].unique()))
    if len(categories) != 2:
        print(f"{col}: variável com {len(categories)} categorias ({categories}), McNemar padrão requer 2. Reportando tabela de contingência:")
        print(pd.crosstab(sub[BASELINE_LABEL], sub[FOLLOWUP_LABEL]))
        continue

    table = pd.crosstab(sub[BASELINE_LABEL], sub[FOLLOWUP_LABEL])
    result = mcnemar(table, exact=True)
    print(f"{col} (N={len(sub)}): estatística={result.statistic:.4f}, p-valor={result.pvalue:.4f}")
    print(table)

=== Checagem de Consistência entre Sessões (variáveis que não deveriam mudar) ===
SEX: 0/38 sujeitos com valores diferentes entre Baseline e Followup_1.
HANDEDNESS_CATEGORY: 0/38 sujeitos com valores diferentes entre Baseline e Followup_1.

=== Teste de McNemar (mudança entre sessões, variáveis categóricas) ===
CURRENT_MED_STATUS (N=38): estatística=3.0000, p-valor=1.0000
Followup_1  0.0  1.0
Baseline            
0.0          23    4
1.0           3    8
EYE_STATUS_AT_SCAN (N=38): estatística=0.0000, p-valor=1.0000
Followup_1  1.0  2.0
Baseline            
1.0          21    0
2.0           0   17


In [ ]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})  # 1 = ASD, 0 = TD
base = base.dropna(subset=['PIQ', 'target', 'SITE_ID'])

print("=== Modelos Específicos por Site: Classificação Intra-Site (PIQ, Leave-One-Out) ===\n")

results_site = []

for site in base['SITE_ID'].unique():
    site_data = base[base['SITE_ID'] == site]
    n = len(site_data)
    n_classes = site_data['target'].nunique()

    if n_classes < 2 or n < 4:
        print(f"Site: {site} (N={n}) - dados insuficientes ou apenas uma classe presente. Análise não realizada.")
        results_site.append({'site': site, 'n': n, 'obs': 'insuficiente'})
        continue

    X = site_data[['PIQ']].values
    y = site_data['target'].values

    loo = LeaveOneOut()
    y_true, y_pred = [], []

    for train_idx, test_idx in loo.split(X):
        y_train = y[train_idx]
        if len(np.unique(y_train)) < 2:
            # Não é possível treinar um classificador com uma única classe no treino
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X[train_idx], y_train)
        pred = clf.predict(X[test_idx])
        y_true.append(y[test_idx][0])
        y_pred.append(pred[0])

    if len(y_true) == 0:
        print(f"Site: {site} (N={n}) - não foi possível treinar (classes insuficientes nos folds).")
        results_site.append({'site': site, 'n': n, 'obs': 'falha no treino'})
        continue

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = (y_true == y_pred).mean()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = np.nanmean([sens, spec])

    n_asd = (y == 1).sum()
    n_td = (y == 0).sum()

    print(f"Site: {site} (N={n}, ASD={n_asd}, TD={n_td})")
    print(f"  Acurácia bruta (LOO intra-site): {acc:.4f}")
    print(f"  Acurácia balanceada: {bal_acc:.4f}")
    print(f"  Sensibilidade (ASD): {sens:.4f} | Especificidade (TD): {spec:.4f}\n")

    results_site.append({
        'site': site, 'n': n, 'n_asd': n_asd, 'n_td': n_td,
        'acc': acc, 'bal_acc': bal_acc, 'sens': sens, 'spec': spec
    })

print("Resumo consolidado:")
print(pd.DataFrame(results_site))

=== Modelos Específicos por Site: Classificação Intra-Site (PIQ, Leave-One-Out) ===

Site: ABIDEI-PITT (N=17, ASD=9, TD=8)
  Acurácia bruta (LOO intra-site): 0.4706
  Acurácia balanceada: 0.4653
  Sensibilidade (ASD): 0.5556 | Especificidade (TD): 0.3750

Site: ABIDEI-UCLA_1 (N=13, ASD=10, TD=3)
  Acurácia bruta (LOO intra-site): 0.6923
  Acurácia balanceada: 0.4500
  Sensibilidade (ASD): 0.9000 | Especificidade (TD): 0.0000

Site: ABIDEI-UCLA_2 (N=8, ASD=4, TD=4)
  Acurácia bruta (LOO intra-site): 0.7500
  Acurácia balanceada: 0.7500
  Sensibilidade (ASD): 0.7500 | Especificidade (TD): 0.7500

Resumo consolidado:
            site   n  n_asd  n_td       acc   bal_acc      sens   spec
0    ABIDEI-PITT  17      9     8  0.470588  0.465278  0.555556  0.375
1  ABIDEI-UCLA_1  13     10     3  0.692308  0.450000  0.900000  0.000
2  ABIDEI-UCLA_2   8      4     4  0.750000  0.750000  0.750000  0.750


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})
base = base.dropna(subset=['PIQ', 'target', 'SITE_ID'])

def loo_balanced_accuracy(X, y):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    for train_idx, test_idx in loo.split(X):
        y_train = y[train_idx]
        if len(np.unique(y_train)) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X[train_idx], y_train)
        pred = clf.predict(X[test_idx])
        y_true.append(y[test_idx][0])
        y_pred.append(pred[0])

    if len(y_true) == 0:
        return np.nan, np.nan

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = (y_true == y_pred).mean()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = np.nanmean([sens, spec])
    return acc, bal_acc

print("=== Teste de Permutação Intra-Site (Leave-One-Out) ===\n")

n_perm = 2000
np.random.seed(42)

for site in base['SITE_ID'].unique():
    site_data = base[base['SITE_ID'] == site]
    n = len(site_data)
    if site_data['target'].nunique() < 2 or n < 4:
        continue

    X = site_data[['PIQ']].values
    y_obs = site_data['target'].values

    obs_acc, obs_bal_acc = loo_balanced_accuracy(X, y_obs)

    perm_accs = np.full(n_perm, np.nan)
    perm_bal_accs = np.full(n_perm, np.nan)

    for i in range(n_perm):
        y_perm = np.random.permutation(y_obs)
        if len(np.unique(y_perm)) < 2:
            continue
        acc_p, bal_acc_p = loo_balanced_accuracy(X, y_perm)
        perm_accs[i] = acc_p
        perm_bal_accs[i] = bal_acc_p

    valid_acc = perm_accs[~np.isnan(perm_accs)]
    valid_bal = perm_bal_accs[~np.isnan(perm_bal_accs)]

    p_acc = (np.sum(valid_acc >= obs_acc) + 1) / (len(valid_acc) + 1)
    p_bal = (np.sum(valid_bal >= obs_bal_acc) + 1) / (len(valid_bal) + 1)

    print(f"Site: {site} (N={n})")
    print(f"  Acurácia observada: {obs_acc:.4f} | p-valor: {p_acc:.4f}")
    print(f"  Acurácia balanceada observada: {obs_bal_acc:.4f} | p-valor: {p_bal:.4f}\n")



=== Teste de Permutação Intra-Site (Leave-One-Out) ===

Site: ABIDEI-PITT (N=17)
  Acurácia observada: 0.4706 | p-valor: 0.4393
  Acurácia balanceada observada: 0.4653 | p-valor: 0.4188

Site: ABIDEI-UCLA_1 (N=13)
  Acurácia observada: 0.6923 | p-valor: 0.8991
  Acurácia balanceada observada: 0.4500 | p-valor: 0.8991

Site: ABIDEI-UCLA_2 (N=8)
  Acurácia observada: 0.7500 | p-valor: 0.1494
  Acurácia balanceada observada: 0.7500 | p-valor: 0.1494

Execute o código e me envie a saída para finalizarmos a interpretação dos modelos por site.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})

features = ['PIQ', 'AGE_AT_SCAN', 'SEX']
base = base.dropna(subset=features + ['target', 'SITE_ID'])

print(f"N final para análise multivariada: {len(base)} sujeitos "
      f"({(base['target']==1).sum()} ASD, {(base['target']==0).sum()} TD)\n")

def leave_site_out_eval_multi(data, feature_cols, label_col, site_col):
    sites = data[site_col].unique()
    y_true_all, y_pred_all = [], []

    for site in sites:
        train = data[data[site_col] != site]
        test = data[data[site_col] == site]

        if train[label_col].nunique() < 2 or len(test) == 0:
            continue

        X_train = train[feature_cols].values
        y_train = train[label_col].values
        X_test = test[feature_cols].values
        y_test = test[label_col].values

        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

    return np.array(y_true_all), np.array(y_pred_all)

def compute_metrics(y_true, y_pred):
    acc = (y_true == y_pred).mean()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = np.nanmean([sens, spec])
    return acc, bal_acc, sens, spec

# --- Modelo Multivariado Observado ---
y_true, y_pred = leave_site_out_eval_multi(base, features, 'target', 'SITE_ID')
obs_acc, obs_bal_acc, obs_sens, obs_spec = compute_metrics(y_true, y_pred)

print("=== Classificação Leave-Site-Out: PIQ + Idade + Sexo ===")
print(f"Acurácia bruta: {obs_acc:.4f}")
print(f"Acurácia balanceada: {obs_bal_acc:.4f}")
print(f"Sensibilidade (ASD): {obs_sens:.4f} | Especificidade (TD): {obs_spec:.4f}")

# --- Teste de Permutação Estratificado por Site ---
n_perm = 2000
np.random.seed(42)
perm_acc = np.zeros(n_perm)
perm_bal_acc = np.zeros(n_perm)

data_perm = base.copy()

for i in range(n_perm):
    data_perm["target_perm"] = (
        data_perm.groupby("SITE_ID")["target"]
        .transform(lambda x: np.random.permutation(x.values))
    )
    y_true_p, y_pred_p = leave_site_out_eval_multi(
        data_perm.drop(columns=["target"]).rename(columns={"target_perm": "target"}),
        features, "target", "SITE_ID"
    )
    if len(y_true_p) == 0:
        perm_acc[i] = np.nan
        perm_bal_acc[i] = np.nan
        continue
    acc_p, bal_acc_p, _, _ = compute_metrics(y_true_p, y_pred_p)
    perm_acc[i] = acc_p
    perm_bal_acc[i] = bal_acc_p

p_acc = (np.nansum(perm_acc >= obs_acc) + 1) / (np.sum(~np.isnan(perm_acc)) + 1)
p_bal_acc = (np.nansum(perm_bal_acc >= obs_bal_acc) + 1) / (np.sum(~np.isnan(perm_bal_acc)) + 1)

print(f"\nP-valor (acurácia bruta): {p_acc:.4f}")
print(f"P-valor (acurácia balanceada): {p_bal_acc:.4f}")

print("\nComparação com o modelo univariado (apenas PIQ):")
print("  PIQ isolado -> Acurácia balanceada: 0.5580 | p-valor: 0.0710")
print(f"  PIQ+Idade+Sexo -> Acurácia balanceada: {obs_bal_acc:.4f} | p-valor: {p_bal_acc:.4f}")


N final para análise multivariada: 38 sujeitos (23 ASD, 15 TD)

=== Classificação Leave-Site-Out: PIQ + Idade + Sexo ===
Acurácia bruta: 0.6579
Acurácia balanceada: 0.6594
Sensibilidade (ASD): 0.6522 | Especificidade (TD): 0.6667

P-valor (acurácia bruta): 0.0750
P-valor (acurácia balanceada): 0.0245

Comparação com o modelo univariado (apenas PIQ):
  PIQ isolado -> Acurácia balanceada: 0.5580 | p-valor: 0.0710
  PIQ+Idade+Sexo -> Acurácia balanceada: 0.6594 | p-valor: 0.0245


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})
base = base.dropna(subset=['PIQ', 'AGE_AT_SCAN', 'SEX', 'target', 'SITE_ID'])

print(f"N = {len(base)} ({(base['target']==1).sum()} ASD, {(base['target']==0).sum()} TD)\n")

# --- 1. Sexo difere entre grupos? (qui-quadrado) ---
print("=== Distribuição de Sexo por Grupo Diagnóstico ===")
ct = pd.crosstab(base['target'], base['SEX'])
print(ct)
chi2, p_chi2, dof, expected = chi2_contingency(ct)
print(f"Qui-quadrado: {chi2:.4f}, p-valor: {p_chi2:.4f}\n")

# --- 2. Idade difere entre grupos? (teste de permutação) ---
def permutation_diff_means(data, feature_col, group_col, n_permutations=2000):
    groups = sorted(data[group_col].unique())
    g1 = data[data[group_col] == groups[0]][feature_col].values
    g2 = data[data[group_col] == groups[1]][feature_col].values
    obs_diff = g1.mean() - g2.mean()

    pooled = data[[feature_col, group_col]].copy()
    diffs = []
    for _ in range(n_permutations):
        perm_labels = np.random.permutation(pooled[group_col].values)
        pg1 = pooled[feature_col][perm_labels == groups[0]]
        pg2 = pooled[feature_col][perm_labels == groups[1]]
        diffs.append(pg1.mean() - pg2.mean())
    diffs = np.array(diffs)
    p_val = (np.sum(np.abs(diffs) >= np.abs(obs_diff)) + 1) / (n_permutations + 1)
    return obs_diff, p_val, g1.mean(), g2.mean()

obs_diff, p_val, mean_asd, mean_td = permutation_diff_means(base, 'AGE_AT_SCAN', 'target')
print("=== Diferença de Idade por Grupo Diagnóstico ===")
print(f"Média ASD: {mean_asd:.2f} | Média TD: {mean_td:.2f}")
print(f"Diferença observada: {obs_diff:.2f} | p-valor (permutação): {p_val:.4f}\n")

# --- 3. Decomposição: cada covariável isolada via Leave-Site-Out ---
def leave_site_out_eval_multi(data, feature_cols, label_col, site_col):
    sites = data[site_col].unique()
    y_true_all, y_pred_all = [], []
    for site in sites:
        train = data[data[site_col] != site]
        test = data[data[site_col] == site]
        if train[label_col].nunique() < 2 or len(test) == 0:
            continue
        X_train = train[feature_cols].values
        y_train = train[label_col].values
        X_test = test[feature_cols].values
        y_test = test[label_col].values
        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
    return np.array(y_true_all), np.array(y_pred_all)

def compute_metrics(y_true, y_pred):
    acc = (y_true == y_pred).mean()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = np.nanmean([sens, spec])
    return acc, bal_acc, sens, spec

def permutation_test_lso(data, feature_cols, n_perm=2000):
    y_true, y_pred = leave_site_out_eval_multi(data, feature_cols, 'target', 'SITE_ID')
    obs_acc, obs_bal_acc, sens, spec = compute_metrics(y_true, y_pred)

    perm_acc = np.zeros(n_perm)
    perm_bal_acc = np.zeros(n_perm)
    data_perm = data.copy()

    for i in range(n_perm):
        data_perm["target_perm"] = (
            data_perm.groupby("SITE_ID")["target"].transform(lambda x: np.random.permutation(x.values))
        )
        yt_p, yp_p = leave_site_out_eval_multi(
            data_perm.drop(columns=["target"]).rename(columns={"target_perm": "target"}),
            feature_cols, "target", "SITE_ID"
        )
        if len(yt_p) == 0:
            perm_acc[i], perm_bal_acc[i] = np.nan, np.nan
            continue
        a, b, _, _ = compute_metrics(yt_p, yp_p)
        perm_acc[i], perm_bal_acc[i] = a, b

    p_acc = (np.nansum(perm_acc >= obs_acc) + 1) / (np.sum(~np.isnan(perm_acc)) + 1)
    p_bal = (np.nansum(perm_bal_acc >= obs_bal_acc) + 1) / (np.sum(~np.isnan(perm_bal_acc)) + 1)
    return obs_acc, obs_bal_acc, sens, spec, p_acc, p_bal

print("=== Decomposição Univariada (Leave-Site-Out) ===\n")
np.random.seed(42)
for feat_set, name in [(['SEX'], 'Sexo isolado'),
                        (['AGE_AT_SCAN'], 'Idade isolada'),
                        (['PIQ'], 'PIQ isolado (referência)'),
                        (['PIQ', 'AGE_AT_SCAN', 'SEX'], 'PIQ+Idade+Sexo (referência)')]:
    acc, bal_acc, sens, spec, p_acc, p_bal = permutation_test_lso(base, feat_set)
    print(f"{name} ({feat_set}):")
    print(f"  Acurácia bruta: {acc:.4f} | Acurácia balanceada: {bal_acc:.4f}")
    print(f"  Sensibilidade: {sens:.4f} | Especificidade: {spec:.4f}")
    print(f"  p-valor (bruta): {p_acc:.4f} | p-valor (balanceada): {p_bal:.4f}\n")


N = 38 (23 ASD, 15 TD)

=== Distribuição de Sexo por Grupo Diagnóstico ===
SEX     1.0  2.0
target          
0        13    2
1        20    3
Qui-quadrado: 0.0000, p-valor: 1.0000

=== Diferença de Idade por Grupo Diagnóstico ===
Média ASD: 13.34 | Média TD: 12.52
Diferença observada: 0.82 | p-valor (permutação): 0.2314

=== Decomposição Univariada (Leave-Site-Out) ===

Sexo isolado (['SEX']):
  Acurácia bruta: 0.6053 | Acurácia balanceada: 0.5000
  Sensibilidade: 1.0000 | Especificidade: 0.0000
  p-valor (bruta): 0.4903 | p-valor (balanceada): 0.4903

Idade isolada (['AGE_AT_SCAN']):
  Acurácia bruta: 0.6579 | Acurácia balanceada: 0.6014
  Sensibilidade: 0.8696 | Especificidade: 0.3333
  p-valor (bruta): 0.1069 | p-valor (balanceada): 0.0975

PIQ isolado (referência) (['PIQ']):
  Acurácia bruta: 0.6053 | Acurácia balanceada: 0.5580
  Sensibilidade: 0.7826 | Especificidade: 0.3333
  p-valor (bruta): 0.1704 | p-valor (balanceada): 0.0690

PIQ+Idade+Sexo (referência) (['PIQ', 'AGE_AT_SC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})
base = base.dropna(subset=['PIQ', 'AGE_AT_SCAN', 'SEX', 'target', 'SITE_ID']).reset_index(drop=True)

def leave_site_out_eval_multi(data, feature_cols, label_col, site_col):
    sites = data[site_col].unique()
    y_true_all, y_pred_all = [], []
    for site in sites:
        train = data[data[site_col] != site]
        test = data[data[site_col] == site]
        if train[label_col].nunique() < 2 or len(test) == 0:
            continue
        X_train = train[feature_cols].values
        y_train = train[label_col].values
        X_test = test[feature_cols].values
        y_test = test[label_col].values
        clf = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
    return np.array(y_true_all), np.array(y_pred_all)

def bal_acc_from_preds(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return np.nanmean([sens, spec])

uni_features = ['PIQ']
multi_features = ['PIQ', 'AGE_AT_SCAN', 'SEX']

# --- Observado ---
yt_uni, yp_uni = leave_site_out_eval_multi(base, uni_features, 'target', 'SITE_ID')
yt_multi, yp_multi = leave_site_out_eval_multi(base, multi_features, 'target', 'SITE_ID')

obs_bal_uni = bal_acc_from_preds(yt_uni, yp_uni)
obs_bal_multi = bal_acc_from_preds(yt_multi, yp_multi)
obs_diff = obs_bal_multi - obs_bal_uni

print(f"Acurácia balanceada - PIQ isolado: {obs_bal_uni:.4f}")
print(f"Acurácia balanceada - PIQ+Idade+Sexo: {obs_bal_multi:.4f}")
print(f"Diferença observada (multi - uni): {obs_diff:.4f}\n")

# --- Teste de Permutação Pareado (mesmos rótulos permutados para os dois modelos) ---
n_perm = 2000
np.random.seed(42)
diffs_perm = np.full(n_perm, np.nan)

data_perm = base.copy()

for i in range(n_perm):
    data_perm["target_perm"] = (
        data_perm.groupby("SITE_ID")["target"].transform(lambda x: np.random.permutation(x.values))
    )
    temp = data_perm.drop(columns=["target"]).rename(columns={"target_perm": "target"})

    yt_u, yp_u = leave_site_out_eval_multi(temp, uni_features, 'target', 'SITE_ID')
    yt_m, yp_m = leave_site_out_eval_multi(temp, multi_features, 'target', 'SITE_ID')

    if len(yt_u) == 0 or len(yt_m) == 0:
        continue

    bal_u = bal_acc_from_preds(yt_u, yp_u)
    bal_m = bal_acc_from_preds(yt_m, yp_m)
    diffs_perm[i] = bal_m - bal_u

valid_diffs = diffs_perm[~np.isnan(diffs_perm)]
p_value_diff = (np.sum(valid_diffs >= obs_diff) + 1) / (len(valid_diffs) + 1)

print(f"Diferença média sob permutação (multi - uni): {np.mean(valid_diffs):.4f}")
print(f"P-valor (ganho do modelo multivariado sobre o univariado): {p_value_diff:.4f}")


Acurácia balanceada - PIQ isolado: 0.5580
Acurácia balanceada - PIQ+Idade+Sexo: 0.6594
Diferença observada (multi - uni): 0.1014

Diferença média sob permutação (multi - uni): 0.0144
P-valor (ganho do modelo multivariado sobre o univariado): 0.1629


## Continuando:  analise Cohen's d

In [ ]:
import pandas as pd
import numpy as np

# --- Carregamento e Limpeza (reutilizado) ---
df = pd.read_csv('ABIDEII_Long_Composite_Phenotypic.csv')
df.columns = [c.strip() for c in df.columns]

sentinel_values = [-9999, -9998, 999, 9999]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_limpo = df.copy()
for col in numeric_cols:
    df_limpo.loc[df_limpo[col].isin(sentinel_values), col] = np.nan

base = df_limpo[df_limpo['SESSION'] == 'Baseline'].copy()
base['target'] = base['DX_GROUP'].map({1: 1, 2: 0})  # 1 = ASD, 0 = TD

def cohens_d(group1, group2):
    """Cohen's d com desvio padrão combinado (pooled)."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    d = (np.mean(group1) - np.mean(group2)) / pooled_std
    return d

def bootstrap_ci_cohens_d(group1, group2, n_boot=5000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    boot_ds = np.zeros(n_boot)
    n1, n2 = len(group1), len(group2)
    g1 = np.array(group1)
    g2 = np.array(group2)

    for i in range(n_boot):
        sample1 = g1[rng.integers(0, n1, n1)]
        sample2 = g2[rng.integers(0, n2, n2)]
        boot_ds[i] = cohens_d(sample1, sample2)

    lower = np.percentile(boot_ds, (100 - ci) / 2)
    upper = np.percentile(boot_ds, 100 - (100 - ci) / 2)
    return lower, upper

def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "desprezível"
    elif ad < 0.5:
        return "pequeno"
    elif ad < 0.8:
        return "médio"
    else:
        return "grande"

print("=== Tamanhos de Efeito (Cohen's d) — ABIDE II Longitudinal (Baseline) ===\n")

for var_name, label in [('PIQ', 'PIQ (QI de Performance)'), ('AGE_AT_SCAN', 'Idade')]:
    data = base.dropna(subset=[var_name, 'target'])
    g_asd = data[data['target'] == 1][var_name].values
    g_td = data[data['target'] == 0][var_name].values

    d = cohens_d(g_asd, g_td)
    ci_low, ci_high = bootstrap_ci_cohens_d(g_asd, g_td)

    print(f"--- {label} ---")
    print(f"N: ASD={len(g_asd)}, TD={len(g_td)}")
    print(f"Média ASD: {np.mean(g_asd):.2f} (DP={np.std(g_asd, ddof=1):.2f})")
    print(f"Média TD: {np.mean(g_td):.2f} (DP={np.std(g_td, ddof=1):.2f})")
    print(f"Cohen's d: {d:.4f} (IC 95% bootstrap: [{ci_low:.4f}, {ci_high:.4f}])")
    print(f"Magnitude do efeito: {interpret_d(d)}\n")



=== Tamanhos de Efeito (Cohen's d) — ABIDE II Longitudinal (Baseline) ===

--- PIQ (QI de Performance) ---
N: ASD=23, TD=15
Média ASD: 102.52 (DP=16.06)
Média TD: 112.87 (DP=9.80)
Cohen's d: -0.7407 (IC 95% bootstrap: [-1.5678, -0.1505])
Magnitude do efeito: médio

--- Idade ---
N: ASD=23, TD=15
Média ASD: 12.52 (DP=2.14)
Média TD: 13.34 (DP=1.82)
Cohen's d: -0.4051 (IC 95% bootstrap: [-1.1652, 0.2215])
Magnitude do efeito: pequeno



## Família 1 — Diferenças descritivas de grupo (baseline): PIQ, Idade, Sexo (3 testes). Família 2 — Classificação Leave-Site-Out (preditiva): PIQ isolado, Idade isolada, Sexo isolado, Multivariado (4 testes). Família 3 — Modelos específicos por sítio (PIQ, LOO intra-sítio): PITT, UCLA_1, UCLA_2 (3 testes).

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

print("=== Correção para Múltiplas Comparações — Estudo de Caso A (ABIDE II Longitudinal) ===\n")

familias = {
    "Família 1 — Diferenças Descritivas de Grupo (Baseline)": {
        "PIQ (ASD vs TD)": 0.035,
        "Idade (ASD vs TD)": 0.2314,
        "Sexo (qui-quadrado)": 1.0000,
    },
    "Família 2 — Classificação Leave-Site-Out (Acurácia Balanceada)": {
        "PIQ isolado": 0.0690,
        "Idade isolada": 0.0975,
        "Sexo isolado": 0.4903,
        "Multivariado (PIQ+Idade+Sexo)": 0.0305,
    },
    "Família 3 — Modelos por Sítio (PIQ, LOO intra-sítio)": {
        "ABIDEI-PITT": 0.4188,
        "ABIDEI-UCLA_1": 0.8991,
        "ABIDEI-UCLA_2": 0.1494,
    },
}

alpha = 0.05
resultados_completos = []

for nome_familia, testes in familias.items():
    print(f"--- {nome_familia} ---")
    labels = list(testes.keys())
    p_values = list(testes.values())

    # Bonferroni
    reject_bonf, pvals_bonf, _, _ = multipletests(p_values, alpha=alpha, method='bonferroni')
    # FDR Benjamini-Hochberg
    reject_fdr, pvals_fdr, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')

    df_familia = pd.DataFrame({
        'Teste': labels,
        'p (bruto)': p_values,
        'p (Bonferroni)': np.round(pvals_bonf, 4),
        'Sig. Bonferroni (α=0.05)': reject_bonf,
        'p (FDR-BH)': np.round(pvals_fdr, 4),
        'Sig. FDR-BH (α=0.05)': reject_fdr,
    })

    print(df_familia.to_string(index=False))
    print()

    resultados_completos.append(df_familia)


=== Correção para Múltiplas Comparações — Estudo de Caso A (ABIDE II Longitudinal) ===

--- Família 1 — Diferenças Descritivas de Grupo (Baseline) ---
              Teste  p (bruto)  p (Bonferroni)  Sig. Bonferroni (α=0.05)  p (FDR-BH)  Sig. FDR-BH (α=0.05)
    PIQ (ASD vs TD)     0.0350          0.1050                     False      0.1050                 False
  Idade (ASD vs TD)     0.2314          0.6942                     False      0.3471                 False
Sexo (qui-quadrado)     1.0000          1.0000                     False      1.0000                 False

--- Família 2 — Classificação Leave-Site-Out (Acurácia Balanceada) ---
                        Teste  p (bruto)  p (Bonferroni)  Sig. Bonferroni (α=0.05)  p (FDR-BH)  Sig. FDR-BH (α=0.05)
                  PIQ isolado     0.0690           0.276                     False      0.1300                 False
                Idade isolada     0.0975           0.390                     False      0.1300                 Fals

# Continuando com Toddlers

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

# --- Carregamento do dataset Q-CHAT-10 Toddler ---
try:
    df_toddler = pd.read_csv('/content/Toddler Autism dataset July 2018.csv')
    print("Arquivo do dataset Toddler carregado com sucesso.")
except FileNotFoundError:
    print("ERRO: Ajuste o nome do arquivo conforme o carregado na sessão (ex: 'Toddler Autism dataset July 2018.csv').")
    raise

print(f"\nColunas disponíveis: {list(df_toddler.columns)}\n")

# --- 1. Balanceamento de Classes ---
print("=== Balanceamento de Classes (Class/ASD Traits) ===")
class_col = [c for c in df_toddler.columns if 'class' in c.lower() or 'asd' in c.lower()]
print(f"Coluna(s) candidata(s) a rótulo: {class_col}")

# Ajuste o nome exato da coluna de rótulo conforme a saída acima, caso necessário
label_col = 'Class/ASD Traits ' if 'Class/ASD Traits ' in df_toddler.columns else class_col[0]
print(df_toddler[label_col].value_counts())
print(f"Proporção: {df_toddler[label_col].value_counts(normalize=True).round(4).to_dict()}\n")

# --- 2. Checagem de valores ausentes ---
print("=== Valores Ausentes por Coluna ===")
print(df_toddler.isna().sum()[df_toddler.isna().sum() > 0])
if df_toddler.isna().sum().sum() == 0:
    print("Nenhum valor ausente encontrado.\n")

# --- 3. V de Cramér e p-valor (com correção) para cada item A1-A10 ---
def cramers_v(confusion_matrix):
    chi2, p, dof, expected = chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)
    v = np.sqrt(phi2_corr / min(k_corr - 1, r_corr - 1))
    return v, chi2, p

def interpret_cramers_v(v, dof_min=1):
    # Interpretação padrão para df=1 (tabelas 2x2, caso dos itens binários A1-A10)
    if v < 0.10:
        return "desprezível"
    elif v < 0.30:
        return "pequeno"
    elif v < 0.50:
        return "médio"
    else:
        return "grande"

item_cols = [c for c in df_toddler.columns if c.startswith('A') and c[1:].split('.')[0].isdigit()]
print(f"\nItens identificados: {item_cols}\n")

print("=== Tamanho de Efeito (V de Cramér) por Item ===")
results = []
for col in item_cols:
    ct = pd.crosstab(df_toddler[col], df_toddler[label_col])
    v, chi2, p = cramers_v(ct)
    results.append({'item': col, 'chi2': chi2, 'p_bruto': p, 'cramers_v': v, 'magnitude': interpret_cramers_v(v)})

df_results = pd.DataFrame(results)

# Correção para múltiplas comparações (Bonferroni e FDR-BH)
reject_bonf, p_bonf, _, _ = multipletests(df_results['p_bruto'], alpha=0.05, method='bonferroni')
reject_fdr, p_fdr, _, _ = multipletests(df_results['p_bruto'], alpha=0.05, method='fdr_bh')

df_results['p_bonferroni'] = p_bonf
df_results['p_fdr_bh'] = p_fdr
df_results['sig_bonferroni'] = reject_bonf
df_results['sig_fdr_bh'] = reject_fdr

print(df_results.to_string(index=False))



Arquivo do dataset Toddler carregado com sucesso.

Colunas disponíveis: ['Case_No', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age_Mons', 'Qchat-10-Score', 'Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who completed the test', 'Class/ASD Traits ']

=== Balanceamento de Classes (Class/ASD Traits) ===
Coluna(s) candidata(s) a rótulo: ['Family_mem_with_ASD', 'Class/ASD Traits ']
Class/ASD Traits 
Yes    728
No     326
Name: count, dtype: int64
Proporção: {'Yes': 0.6907, 'No': 0.3093}

=== Valores Ausentes por Coluna ===
Series([], dtype: int64)
Nenhum valor ausente encontrado.


Itens identificados: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10']

=== Tamanho de Efeito (V de Cramér) por Item ===
item       chi2      p_bruto  cramers_v magnitude  p_bonferroni     p_fdr_bh  sig_bonferroni  sig_fdr_bh
  A1 265.337325 1.178016e-59   0.501031    grande  1.178016e-58 1.963360e-59            True        True
  A2 224.389727 9.975027e-51   0.460592     m

In [ ]:
label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

print("=== Distribuição de Respostas por Item (0 vs 1) ===\n")
for col in item_cols:
    counts = df_toddler[col].value_counts(normalize=True).sort_index()
    print(f"{col}: {dict(counts.round(4))}")

print("\n=== Correlação de Cada Item com a Soma dos Demais 9 Itens (Correlação Item-Restante) ===")
score_col = 'Qchat-10-Score'

for col in item_cols:
    rest_sum = df_toddler[item_cols].drop(columns=[col]).sum(axis=1)
    corr = df_toddler[col].corr(rest_sum)
    print(f"{col}: correlação item-restante = {corr:.4f}")

print("\n=== Verificação: Score é de fato a soma direta de A1-A10? ===")
soma_manual = df_toddler[item_cols].sum(axis=1)
diferenca = (soma_manual - df_toddler[score_col]).abs().sum()
print(f"Soma das diferenças absolutas entre Score reportado e soma manual dos itens: {diferenca}")
print(f"(Deve ser 0 ou próximo de 0 se Score = soma direta dos itens)")

=== Distribuição de Respostas por Item (0 vs 1) ===

A1: {0: np.float64(0.4364), 1: np.float64(0.5636)}
A2: {0: np.float64(0.5512), 1: np.float64(0.4488)}
A3: {0: np.float64(0.5987), 1: np.float64(0.4013)}
A4: {0: np.float64(0.4877), 1: np.float64(0.5123)}
A5: {0: np.float64(0.4753), 1: np.float64(0.5247)}
A6: {0: np.float64(0.4231), 1: np.float64(0.5769)}
A7: {0: np.float64(0.3501), 1: np.float64(0.6499)}
A8: {0: np.float64(0.5408), 1: np.float64(0.4592)}
A9: {0: np.float64(0.5104), 1: np.float64(0.4896)}
A10: {0: np.float64(0.4137), 1: np.float64(0.5863)}

=== Correlação de Cada Item com a Soma dos Demais 9 Itens (Correlação Item-Restante) ===
A1: correlação item-restante = 0.4898
A2: correlação item-restante = 0.4634
A3: correlação item-restante = 0.4661
A4: correlação item-restante = 0.5237
A5: correlação item-restante = 0.5320
A6: correlação item-restante = 0.5455
A7: correlação item-restante = 0.5012
A8: correlação item-restante = 0.4049
A9: correlação item-restante = 0.5591
A10:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import LeaveOneOut, cross_val_score


label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

y = (df_toddler[label_col] == 'Yes').astype(int)

def loocv_accuracy(X, y):
    clf = GaussianNB()
    loo = LeaveOneOut()
    scores = cross_val_score(clf, X, y, cv=loo, scoring='accuracy')
    return scores.mean()

print("=== Comparação: Classificação com Todos os Itens vs. Excluindo A10 ===\n")

X_all = df_toddler[item_cols].values
acc_all = loocv_accuracy(X_all, y)
print(f"Acurácia (A1-A10 completos, LOOCV): {acc_all:.4f}")

item_cols_no_a10 = [c for c in item_cols if c != 'A10']
X_no_a10 = df_toddler[item_cols_no_a10].values
acc_no_a10 = loocv_accuracy(X_no_a10, y)
print(f"Acurácia (A1-A9, excluindo A10, LOOCV): {acc_no_a10:.4f}")

print(f"\nDiferença (completo - sem A10): {acc_all - acc_no_a10:.4f}")

=== Comparação: Classificação com Todos os Itens vs. Excluindo A10 ===

Acurácia (A1-A10 completos, LOOCV): 0.9440
Acurácia (A1-A9, excluindo A10, LOOCV): 0.9374

Diferença (completo - sem A10): 0.0066


In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB, BernoulliNB, CategoricalNB
from sklearn.model_selection import LeaveOneOut, cross_val_score

label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

y = (df_toddler[label_col] == 'Yes').astype(int)
X_all = df_toddler[item_cols].values

def loocv_accuracy(clf, X, y):
    loo = LeaveOneOut()
    scores = cross_val_score(clf, X, y, cv=loo, scoring='accuracy')
    return scores.mean()

print("=== Comparação de Classificadores Naive Bayes (A1-A10, sem Score) ===\n")

for name, clf in [('GaussianNB', GaussianNB()),
                   ('BernoulliNB', BernoulliNB()),
                   ('CategoricalNB', CategoricalNB())]:
    acc = loocv_accuracy(clf, X_all, y)
    print(f"{name}: acurácia LOOCV = {acc:.4f}")

print("\n=== Comparação com/sem A10, usando o classificador mais apropriado (BernoulliNB) ===")
item_cols_no_a10 = [c for c in item_cols if c != 'A10']
X_no_a10 = df_toddler[item_cols_no_a10].values

acc_all_bnb = loocv_accuracy(BernoulliNB(), X_all, y)
acc_no_a10_bnb = loocv_accuracy(BernoulliNB(), X_no_a10, y)

print(f"BernoulliNB (A1-A10 completos): {acc_all_bnb:.4f}")
print(f"BernoulliNB (A1-A9, sem A10): {acc_no_a10_bnb:.4f}")
print(f"Diferença: {acc_all_bnb - acc_no_a10_bnb:.4f}")



=== Comparação de Classificadores Naive Bayes (A1-A10, sem Score) ===

GaussianNB: acurácia LOOCV = 0.9440
BernoulliNB: acurácia LOOCV = 0.9554
CategoricalNB: acurácia LOOCV = 0.9554

=== Comparação com/sem A10, usando o classificador mais apropriado (BernoulliNB) ===
BernoulliNB (A1-A10 completos): 0.9554
BernoulliNB (A1-A9, sem A10): 0.9412
Diferença: 0.0142


In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import LabelEncoder

df_toddler = pd.read_csv('/content/Toddler Autism dataset July 2018.csv')
label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

y = (df_toddler[label_col] == 'Yes').astype(int)

def loocv_accuracy(clf, X, y):
    loo = LeaveOneOut()
    scores = cross_val_score(clf, X, y, cv=loo, scoring='accuracy')
    return scores.mean()

print("=== Hipótese 1: Modelo original incluiu variáveis demográficas além de A1-A10? ===\n")

demo_cols = ['Age_Mons', 'Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who completed the test']
print(f"Colunas demográficas candidatas: {demo_cols}\n")

df_encoded = df_toddler.copy()
for col in demo_cols:
    if df_encoded[col].dtype == 'object':
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col].astype(str))

# Confirmar que Score NÃO está em nenhuma dessas listas
assert 'Qchat-10-Score' not in item_cols + demo_cols, "ALERTA: Score está incluído nas features!"
print("Confirmado: 'Qchat-10-Score' não está incluído em nenhum dos conjuntos de features testados.\n")

X_items_only = df_encoded[item_cols].values
X_items_demo = df_encoded[item_cols + demo_cols].values

print("--- Usando BernoulliNB ---")
print(f"Apenas A1-A10: {loocv_accuracy(BernoulliNB(), X_items_only, y):.4f}")
print(f"A1-A10 + Demográficas: {loocv_accuracy(BernoulliNB(), X_items_demo, y):.4f}\n")

print("=== Hipótese 2: Modelo original era Árvore de Decisão ou Random Forest? ===\n")

for name, clf in [('DecisionTree (A1-A10)', DecisionTreeClassifier(random_state=42)),
                   ('RandomForest (A1-A10)', RandomForestClassifier(random_state=42, n_estimators=100))]:
    acc = loocv_accuracy(clf, X_items_only, y)
    print(f"{name}: acurácia LOOCV = {acc:.4f}")

print()
for name, clf in [('DecisionTree (A1-A10+Demo)', DecisionTreeClassifier(random_state=42)),
                   ('RandomForest (A1-A10+Demo)', RandomForestClassifier(random_state=42, n_estimators=100))]:
    acc = loocv_accuracy(clf, X_items_demo, y)
    print(f"{name}: acurácia LOOCV = {acc:.4f}")

print("\n=== Hipótese 3: Alguma variável demográfica isolada tem alta correlação com Class (vazamento residual)? ===\n")
from scipy.stats import chi2_contingency

for col in demo_cols:
    if col == 'Age_Mons':
        continue  # variável contínua, tratada à parte se necessário
    ct = pd.crosstab(df_toddler[col], df_toddler[label_col])
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"{col}: chi2={chi2:.4f}, p={p:.6f}")


=== Hipótese 1: Modelo original incluiu variáveis demográficas além de A1-A10? ===

Colunas demográficas candidatas: ['Age_Mons', 'Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who completed the test']

Confirmado: 'Qchat-10-Score' não está incluído em nenhum dos conjuntos de features testados.

--- Usando BernoulliNB ---
Apenas A1-A10: 0.9554
A1-A10 + Demográficas: 0.9611

=== Hipótese 2: Modelo original era Árvore de Decisão ou Random Forest? ===

DecisionTree (A1-A10): acurácia LOOCV = 0.9583
RandomForest (A1-A10): acurácia LOOCV = 0.9687

DecisionTree (A1-A10+Demo): acurácia LOOCV = 0.9250
RandomForest (A1-A10+Demo): acurácia LOOCV = 0.9573

=== Hipótese 3: Alguma variável demográfica isolada tem alta correlação com Class (vazamento residual)? ===

Sex: chi2=14.0436, p=0.000179
Ethnicity: chi2=43.5713, p=0.000004
Jaundice: chi2=5.4270, p=0.019827
Family_mem_with_ASD: chi2=0.1209, p=0.728014
Who completed the test: chi2=3.7881, p=0.435448


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut


label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

y = (df_toddler[label_col] == 'Yes').astype(int).values
X_full = df_toddler[item_cols].values
n_top = 3

# --- Versão 1: Seleção de Features VAZADA (réplica do procedimento original) ---
# Seleciona os top-3 itens usando correlação calculada no dataset INTEIRO,
# depois roda LOOCV apenas com esses 3 itens fixos.
correlacoes_full = [np.corrcoef(df_toddler[item].values, y)[0, 1] for item in item_cols]
ranking_full = pd.Series(correlacoes_full, index=item_cols).sort_values(ascending=False)
top3_leaky = ranking_full.head(n_top).index.tolist()
print(f"Top-3 itens selecionados usando o dataset completo (vazado): {top3_leaky}")

X_leaky = df_toddler[top3_leaky].values
loo = LeaveOneOut()
preds_leaky = []
for train_idx, test_idx in loo.split(X_leaky):
    clf = LogisticRegression()
    clf.fit(X_leaky[train_idx], y[train_idx])
    preds_leaky.append(clf.predict(X_leaky[test_idx])[0])
acc_leaky = (np.array(preds_leaky) == y).mean()
print(f"Acurácia LOOCV (seleção vazada, fixa): {acc_leaky:.4f}\n")

# --- Versão 2: Seleção de Features CORRETA (aninhada dentro do LOOCV) ---
# A cada fold, a correlação e o ranking dos top-3 itens são recalculados
# usando APENAS os dados de treino daquele fold.
preds_nested = []
itens_selecionados_por_fold = []

for train_idx, test_idx in loo.split(X_full):
    X_train, y_train = X_full[train_idx], y[train_idx]
    X_test = X_full[test_idx]

    correlacoes_train = [np.corrcoef(X_train[:, i], y_train)[0, 1] for i in range(len(item_cols))]
    ranking_train = pd.Series(correlacoes_train, index=item_cols).sort_values(ascending=False)
    top3_train = ranking_train.head(n_top).index.tolist()
    itens_selecionados_por_fold.append(tuple(sorted(top3_train)))

    idx_top3 = [item_cols.index(item) for item in top3_train]
    clf = LogisticRegression()
    clf.fit(X_train[:, idx_top3], y_train)
    preds_nested.append(clf.predict(X_test[:, idx_top3])[0])

acc_nested = (np.array(preds_nested) == y).mean()
print(f"Acurácia LOOCV (seleção aninhada, correta): {acc_nested:.4f}")

from collections import Counter
contagem_selecoes = Counter(itens_selecionados_por_fold)
print(f"\nNúmero de combinações distintas de top-3 itens ao longo dos {len(y)} folds: {len(contagem_selecoes)}")
print("As 5 combinações mais frequentes:")
for combo, count in contagem_selecoes.most_common(5):
    print(f"  {combo}: {count} folds ({count/len(y)*100:.1f}%)")

print(f"\nDiferença (vazado - aninhado): {acc_leaky - acc_nested:.4f}")



Top-3 itens selecionados usando o dataset completo (vazado): ['A9', 'A6', 'A5']
Acurácia LOOCV (seleção vazada, fixa): 0.8719

Acurácia LOOCV (seleção aninhada, correta): 0.8368

Número de combinações distintas de top-3 itens ao longo dos 1054 folds: 2
As 5 combinações mais frequentes:
  ('A5', 'A6', 'A9'): 858 folds (81.4%)
  ('A6', 'A7', 'A9'): 196 folds (18.6%)

Diferença (vazado - aninhado): 0.0351


# Toddlers Part 2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB, BernoulliNB, CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix

df_toddler = pd.read_csv('/content/Toddler Autism dataset July 2018.csv')
label_col = 'Class/ASD Traits '
item_cols = [f'A{i}' for i in range(1, 11)]

y = (df_toddler[label_col] == 'Yes').astype(int).values
X = df_toddler[item_cols].values

def loocv_balanced_accuracy(clf, X, y):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    for train_idx, test_idx in loo.split(X):
        clf.fit(X[train_idx], y[train_idx])
        y_pred.append(clf.predict(X[test_idx])[0])
        y_true.append(y[test_idx][0])
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = (y_true == y_pred).mean()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = np.nanmean([sens, spec])
    return acc, bal_acc, sens, spec

print("=== Acurácia Bruta vs. Balanceada (A1-A10, sem Score, LOOCV) ===\n")
for name, clf in [('GaussianNB', GaussianNB()),
                   ('BernoulliNB', BernoulliNB()),
                   ('CategoricalNB', CategoricalNB()),
                   ('DecisionTree', DecisionTreeClassifier(random_state=42)),
                   ('RandomForest', RandomForestClassifier(random_state=42, n_estimators=100))]:
    acc, bal_acc, sens, spec = loocv_balanced_accuracy(clf, X, y)
    print(f"{name}: Acc={acc:.4f} | Acc.Bal={bal_acc:.4f} | Sens={sens:.4f} | Espec={spec:.4f}")

=== Acurácia Bruta vs. Balanceada (A1-A10, sem Score, LOOCV) ===

GaussianNB: Acc=0.9440 | Acc.Bal=0.9358 | Sens=0.9574 | Espec=0.9141
BernoulliNB: Acc=0.9554 | Acc.Bal=0.9669 | Sens=0.9368 | Espec=0.9969
CategoricalNB: Acc=0.9554 | Acc.Bal=0.9669 | Sens=0.9368 | Espec=0.9969
DecisionTree: Acc=0.9583 | Acc.Bal=0.9537 | Sens=0.9657 | Espec=0.9417
RandomForest: Acc=0.9687 | Acc.Bal=0.9638 | Sens=0.9766 | Espec=0.9509
